In [2]:
!pip install -q datasets scikit-learn mlflow joblib

In [3]:
import requests, json, random
import pandas as pd

random.seed(42)

URL = ("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/"
       "resolve/main/raw/review_categories/Video_Games.jsonl")

CUTOFF_MS = 1577836799000   # 31 Des 2019 23:59:59 UTC (timestamp dalam milidetik)
SAMPLE_P  = 0.08            # ambil ~8% baris secara acak
MAX_ROWS  = 400_000

rows = []
with requests.get(URL, stream=True, timeout=60) as resp:
    resp.raise_for_status()
    for line in resp.iter_lines():          # dibaca satu baris, gak dimuat utuh
        if not line:
            continue
        r = json.loads(line)
        if r["timestamp"] > CUTOFF_MS:    continue
        if r["rating"] == 3:              continue
        if random.random() > SAMPLE_P:    continue
        title = r.get("title") or ""
        text  = r.get("text") or ""
        rows.append({
            "text": f"{title} {text}".strip(),
            "label": 1 if r["rating"] >= 4 else 0,
            "timestamp": r["timestamp"],
        })
        if len(rows) >= MAX_ROWS:
            break

df = pd.DataFrame(rows)
print(df.shape)
print(df["label"].value_counts(normalize=True))

(226060, 3)
label
1    0.824432
0    0.175568
Name: proportion, dtype: float64


In [4]:
# 1. Bersihin: buang teks kosong dan duplikat (biar gak bocor ke validasi)
df = df[df["text"].str.len() > 0]
df = df.drop_duplicates(subset="text").reset_index(drop=True)

# 2. Cek sebaran per tahun
df["year"] = pd.to_datetime(df["timestamp"], unit="ms").dt.year
print(df.groupby("year")["label"].agg(jumlah="count", persen_positif="mean"))

# 3. Split berdasarkan waktu: 80% terlama = train, 20% terbaru = validasi
df = df.sort_values("timestamp").reset_index(drop=True)
cut = int(len(df) * 0.8)
train, val = df.iloc[:cut], df.iloc[cut:]

print("\ntrain:", train.shape, "| validasi:", val.shape)
print("proporsi positif train:", round(train["label"].mean(), 3))
print("proporsi positif val  :", round(val["label"].mean(), 3))
print("train s/d :", pd.to_datetime(train["timestamp"].max(), unit="ms"))
print("val mulai :", pd.to_datetime(val["timestamp"].min(), unit="ms"))

# 4. Simpen biar kalau Colab restart gak perlu download ulang (±3 menit)
df.to_parquet("data_sample.parquet")

      jumlah  persen_positif
year                        
1998       1        1.000000
1999      39        0.871795
2000     361        0.842105
2001     782        0.826087
2002    1086        0.826888
2003    1017        0.830875
2004    1111        0.798380
2005    1107        0.796748
2006    1176        0.764456
2007    2077        0.810785
2008    2701        0.778971
2009    3684        0.828719
2010    4607        0.796180
2011    6539        0.799205
2012    8185        0.814661
2013   18119        0.829847
2014   21695        0.839226
2015   26439        0.822081
2016   25844        0.816708
2017   24639        0.787816
2018   26698        0.789460
2019   32601        0.818748

train: (168406, 4) | validasi: (42102, 4)
proporsi positif train: 0.813
proporsi positif val  : 0.813
train s/d : 2018-08-22 12:33:02.870000
val mulai : 2018-08-22 13:18:55.246000


In [5]:
import os, joblib, mlflow
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

os.makedirs("models", exist_ok=True)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("sentimen-video-games")

params = {"ngram_range": (1, 2), "min_df": 5, "max_features": 200_000, "C": 1.0}

model_v1 = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=params["ngram_range"],
                              min_df=params["min_df"],
                              max_features=params["max_features"],
                              sublinear_tf=True)),
    ("clf", LogisticRegression(C=params["C"], class_weight="balanced",
                               max_iter=300)),
])

with mlflow.start_run(run_name="v1_tfidf_logreg"):
    model_v1.fit(train["text"], train["label"])
    pred = model_v1.predict(val["text"])

    macro_f1 = f1_score(val["label"], pred, average="macro")
    f1_neg   = f1_score(val["label"], pred, pos_label=0)
    f1_pos   = f1_score(val["label"], pred, pos_label=1)

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metrics({"macro_f1": macro_f1, "f1_negatif": f1_neg, "f1_positif": f1_pos})
    mlflow.set_tag("version", "v1")

    joblib.dump(model_v1, "models/model_v1.pkl")
    mlflow.log_artifact("models/model_v1.pkl")

print("macro-F1 v1:", round(macro_f1, 4))
print(classification_report(val["label"], pred, target_names=["negatif", "positif"]))
print("ukuran file (MB):", round(os.path.getsize("models/model_v1.pkl") / 1e6, 1))

2026/11/20 09:20:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/20 09:20:03 INFO mlflow.store.db.utils: Updating database tables
2026/11/20 09:20:10 INFO mlflow.tracking.fluent: Experiment with name 'sentimen-video-games' does not exist. Creating a new experiment.


macro-F1 v1: 0.9262
              precision    recall  f1-score   support

     negatif       0.83      0.94      0.88      7889
     positif       0.99      0.95      0.97     34213

    accuracy                           0.95     42102
   macro avg       0.91      0.95      0.93     42102
weighted avg       0.96      0.95      0.95     42102

ukuran file (MB): 9.6


In [6]:
from sklearn.metrics import f1_score

# 1. Pisahin holdout dari train (tetap berdasarkan waktu)
cut2 = int(len(train) * 0.85)
core, hold = train.iloc[:cut2], train.iloc[cut2:]

# 2. TF-IDF di-fit sekali aja biar cepat
vec = TfidfVectorizer(ngram_range=params["ngram_range"], min_df=params["min_df"],
                      max_features=params["max_features"], sublinear_tf=True)
Xc = vec.fit_transform(core["text"])
Xh = vec.transform(hold["text"])

# 3. Coba beberapa setting, catat semuanya di MLflow
results = []
for cw in ["balanced", None]:
    for C in [1, 3, 10]:
        clf = LogisticRegression(C=C, class_weight=cw, max_iter=500)
        clf.fit(Xc, core["label"])
        f = f1_score(hold["label"], clf.predict(Xh), average="macro")
        with mlflow.start_run(run_name=f"tuning_C{C}_cw{cw}"):
            mlflow.log_params({"C": C, "class_weight": str(cw)})
            mlflow.log_metric("macro_f1_holdout", f)
            mlflow.set_tag("stage", "tuning")
        results.append((f, C, cw))
        print(f"C={C:<3} class_weight={str(cw):<9} macro-F1 holdout = {f:.4f}")

best_f, best_C, best_cw = max(results, key=lambda x: x[0])
print("\nTerbaik:", "C =", best_C, "| class_weight =", best_cw)

# 4. Latih ulang di SELURUH train dengan setting terbaik, lalu uji di validasi (sekali aja)
model_v2 = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=params["ngram_range"], min_df=params["min_df"],
                              max_features=params["max_features"], sublinear_tf=True)),
    ("clf", LogisticRegression(C=best_C, class_weight=best_cw, max_iter=500)),
])

with mlflow.start_run(run_name="v2_tuned"):
    model_v2.fit(train["text"], train["label"])
    pred2 = model_v2.predict(val["text"])

    macro_f1_v2 = f1_score(val["label"], pred2, average="macro")
    mlflow.log_params({"C": best_C, "class_weight": str(best_cw),
                       "ngram_range": str(params["ngram_range"]),
                       "min_df": params["min_df"], "max_features": params["max_features"]})
    mlflow.log_metrics({"macro_f1": macro_f1_v2,
                        "f1_negatif": f1_score(val["label"], pred2, pos_label=0),
                        "f1_positif": f1_score(val["label"], pred2, pos_label=1)})
    mlflow.set_tag("version", "v2")

    joblib.dump(model_v2, "models/model_v2.pkl")
    mlflow.log_artifact("models/model_v2.pkl")

print("\nmacro-F1 v1:", round(macro_f1, 4))
print("macro-F1 v2:", round(macro_f1_v2, 4))
print(classification_report(val["label"], pred2, target_names=["negatif", "positif"]))
print("ukuran file v2 (MB):", round(os.path.getsize("models/model_v2.pkl") / 1e6, 1))

C=1   class_weight=balanced  macro-F1 holdout = 0.9381
C=3   class_weight=balanced  macro-F1 holdout = 0.9427
C=10  class_weight=balanced  macro-F1 holdout = 0.9446
C=1   class_weight=None      macro-F1 holdout = 0.9303
C=3   class_weight=None      macro-F1 holdout = 0.9384
C=10  class_weight=None      macro-F1 holdout = 0.9441

Terbaik: C = 10 | class_weight = balanced

macro-F1 v1: 0.9262
macro-F1 v2: 0.9345
              precision    recall  f1-score   support

     negatif       0.86      0.93      0.89      7889
     positif       0.98      0.97      0.97     34213

    accuracy                           0.96     42102
   macro avg       0.92      0.95      0.93     42102
weighted avg       0.96      0.96      0.96     42102

ukuran file v2 (MB): 9.6


In [7]:
import json
info = {
    "active_version": "v2",
    "models": {
        "v1": {"file": "models/model_v1.pkl", "macro_f1": round(float(macro_f1), 4)},
        "v2": {"file": "models/model_v2.pkl", "macro_f1": round(float(macro_f1_v2), 4)},
    },
}
json.dump(info, open("models/model_info.json", "w"), indent=2)
print(info)

{'active_version': 'v2', 'models': {'v1': {'file': 'models/model_v1.pkl', 'macro_f1': 0.9262}, 'v2': {'file': 'models/model_v2.pkl', 'macro_f1': 0.9345}}}


In [8]:
%%writefile app.py
import json, time
from pathlib import Path
import joblib
from fastapi import FastAPI
from pydantic import BaseModel

BASE = Path(__file__).parent
INFO = json.loads((BASE / "models" / "model_info.json").read_text())
VERSION = INFO["active_version"]
model = joblib.load(BASE / INFO["models"][VERSION]["file"])
LOG_FILE = BASE / "predictions.log"

app = FastAPI(title="Sentimen Ulasan Video Games")

class Review(BaseModel):
    title: str = ""
    text: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/model-info")
def model_info():
    return INFO

@app.post("/predict")
def predict(review: Review):
    # format input HARUS sama dengan saat training: "judul teks"
    teks = f"{review.title} {review.text}".strip()
    proba = model.predict_proba([teks])[0]
    pred = int(proba.argmax())
    hasil = {
        "label": "positif" if pred == 1 else "negatif",
        "probability": round(float(proba[pred]), 4),
        "model_version": VERSION,
    }
    # observability: catat tiap prediksi
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps({"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                            "input": teks[:500], **hasil}) + "\n")
    return hasil

Writing app.py


In [9]:
!pip install -q fastapi httpx uvicorn
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)
print(client.get("/health").json())
print(client.get("/model-info").json())
print(client.post("/predict", json={"title": "Waste of money",
      "text": "Crashes every 5 minutes, total garbage."}).json())
print(client.post("/predict", json={"title": "Amazing!",
      "text": "Best game I have played in years, love it."}).json())
!tail -n 2 predictions.log

{'status': 'ok'}
{'active_version': 'v2', 'models': {'v1': {'file': 'models/model_v1.pkl', 'macro_f1': 0.9262}, 'v2': {'file': 'models/model_v2.pkl', 'macro_f1': 0.9345}}}
{'label': 'negatif', 'probability': 1.0, 'model_version': 'v2'}
{'label': 'positif', 'probability': 1.0, 'model_version': 'v2'}
{"waktu": "2026-11-20 09:23:56", "input": "Waste of money Crashes every 5 minutes, total garbage.", "label": "negatif", "probability": 1.0, "model_version": "v2"}
{"waktu": "2026-11-20 09:23:56", "input": "Amazing! Best game I have played in years, love it.", "label": "positif", "probability": 1.0, "model_version": "v2"}


In [10]:
%%writefile retrain.py
import json
from pathlib import Path
import joblib, mlflow
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

INFO_PATH = Path("models/model_info.json")

def _score(model, df):
    return f1_score(df["label"], model.predict(df["text"]), average="macro")

def retrain_if_needed(new_df, train_df, threshold=0.85, dry_run=False):
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("sentimen-video-games")

    info = json.loads(INFO_PATH.read_text())
    active = info["active_version"]
    old_model = joblib.load(info["models"][active]["file"])

    # bagi data baru: 80% buat belajar lagi, 20% buat uji adil lama vs baru
    cut = int(len(new_df) * 0.8)
    new_learn, new_test = new_df.iloc[:cut], new_df.iloc[cut:]
    old_score = _score(old_model, new_test)

    with mlflow.start_run(run_name=f"monitoring_{active}"):
        mlflow.log_metric("macro_f1_data_baru", old_score)
        mlflow.set_tag("stage", "monitoring")

    print(f"Model aktif {active} | macro-F1 di data baru = {old_score:.4f} | batas = {threshold}")
    if old_score >= threshold:
        print("Performa aman, tidak perlu re-training.")
        return {"retrained": False, "score": round(old_score, 4)}

    print("Performa di bawah batas -> perlu re-training.")
    if dry_run:
        print("(dry_run: re-training tidak dijalankan)")
        return {"retrained": "dry_run", "score": round(old_score, 4)}

    new_version = f"v{max(int(v[1:]) for v in info['models']) + 1}"
    data = pd.concat([train_df, new_learn])
    model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=5,
                                  max_features=200_000, sublinear_tf=True)),
        ("clf", LogisticRegression(C=10, class_weight="balanced", max_iter=500)),
    ])
    model.fit(data["text"], data["label"])
    new_score = _score(model, new_test)

    file = f"models/model_{new_version}.pkl"
    joblib.dump(model, file)
    with mlflow.start_run(run_name=f"{new_version}_retrain"):
        mlflow.log_metrics({"macro_f1": new_score, "macro_f1_model_lama": old_score})
        mlflow.set_tag("version", new_version)
        mlflow.log_artifact(file)

    info["models"][new_version] = {"file": file, "macro_f1": round(new_score, 4)}
    if new_score > old_score:
        info["active_version"] = new_version
        print(f"{new_version} lebih bagus ({new_score:.4f}) -> jadi model aktif.")
    else:
        print(f"{new_version} tidak lebih bagus ({new_score:.4f}) -> model lama tetap dipakai.")
    INFO_PATH.write_text(json.dumps(info, indent=2))
    return {"retrained": True, "version": new_version, "score": round(new_score, 4)}

Writing retrain.py


In [11]:
from retrain import retrain_if_needed

# Skenario 1: kondisi normal -> harusnya "tidak perlu re-training"
print(retrain_if_needed(val, train, threshold=0.85))
print()

# Skenario 2: simulasi ada masalah (batas dinaikin jadi 0.95), pakai dry_run
print(retrain_if_needed(val, train, threshold=0.95, dry_run=True))

Model aktif v2 | macro-F1 di data baru = 0.9367 | batas = 0.85
Performa aman, tidak perlu re-training.
{'retrained': False, 'score': 0.9367}

Model aktif v2 | macro-F1 di data baru = 0.9367 | batas = 0.95
Performa di bawah batas -> perlu re-training.
(dry_run: re-training tidak dijalankan)
{'retrained': 'dry_run', 'score': 0.9367}


In [12]:
from importlib.metadata import version

pkgs = ["numpy", "scipy", "pandas", "pyarrow", "scikit-learn", "joblib",
        "mlflow", "fastapi", "uvicorn", "httpx", "requests"]
with open("requirements.txt", "w") as f:
    for p in pkgs:
        f.write(f"{p}=={version(p)}\n")
print(open("requirements.txt").read())

numpy==2.4.6
scipy==1.16.3
pandas==2.2.3
pyarrow==23.0.1
scikit-learn==1.6.1
joblib==1.6.0
mlflow==3.18.0
fastapi==0.141.1
uvicorn==0.54.0
httpx==0.28.1
requests==2.32.4



In [13]:
%%writefile .gitignore
__pycache__/
.ipynb_checkpoints/
sample_data/
.config/
data_sample.parquet
predictions.log
*.zip

Writing .gitignore


In [14]:
!zip -rq hasil.zip app.py retrain.py requirements.txt .gitignore models mlruns mlflow.db
from google.colab import files
files.download("hasil.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from importlib.metadata import version

api_pkgs = ["numpy", "scipy", "scikit-learn", "joblib", "fastapi", "uvicorn"]
with open("requirements-render.txt", "w") as f:
    for p in api_pkgs:
        f.write(f"{p}=={version(p)}\n")
print(open("requirements-render.txt").read())

numpy==2.4.6
scipy==1.16.3
scikit-learn==1.6.1
joblib==1.6.0
fastapi==0.141.1
uvicorn==0.54.0

